# Amila (CCSD(T)) vs Experimental RPFR Comparison

This notebook compares RPFR contributions computed from Amila's CCSD(T) molecular
constants against those derived from experimental spectroscopic constants.

For each isotopologue pair the comparison covers:
- Translational
- ZPE (total)
- Excited state (harmonic & anharmonic)
- Rotational (diatomic)
- Rotational-vibrational coupling
- Total RPFR at 273.15 K

In [10]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Ensure the package is importable
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from richet_rpfr import (
    load_experimental_constants,
    load_amila_rpfr,
    generate_all_variants,
    compute_experimental_rpfr,
    compare_amila_to_experimental,
    AMILA_CONTRIB_COLS,
)

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

## 1. Load data

In [11]:
DATA_FILE = project_root / "RPFR-CCSD(T)_DTQ (1).xlsx"

# Amila's computed values
amila_df = load_amila_rpfr(DATA_FILE)

# Experimental constants + isotopic variants
exp_parents = load_experimental_constants(DATA_FILE)
exp_lookup = generate_all_variants(exp_parents)

print(f"Amila pairs loaded: {len(amila_df)}")
print(f"Experimental isotopologues available: {len(exp_lookup)}")

Amila pairs loaded: 131
Experimental isotopologues available: 172


## 2. Compute RPFR from experimental constants

In [12]:
exp_rpfr_df = compute_experimental_rpfr(amila_df, exp_lookup, temperature=273.15)

print(f"Experimental RPFR computed for {len(exp_rpfr_df)} pairs")
print(f"Pairs with NaN RPFR: {exp_rpfr_df['RPFR 273.15K'].isna().sum()}")

Experimental RPFR computed for 131 pairs
Pairs with NaN RPFR: 0


## 3. Write experimental RPFR to Excel

Saves the experimental RPFR as a new sheet in the workbook.

In [13]:
from openpyxl import load_workbook

wb = load_workbook(DATA_FILE)
sheet_name = "Experimental RPFR"
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
wb.save(DATA_FILE)

with pd.ExcelWriter(DATA_FILE, engine="openpyxl", mode="a") as writer:
    exp_rpfr_df[["Molecule", "Method"] + AMILA_CONTRIB_COLS].to_excel(
        writer, sheet_name=sheet_name, index=False
    )

print(f"Written sheet '{sheet_name}' to {DATA_FILE.name}")

Written sheet 'Experimental RPFR' to RPFR-CCSD(T)_DTQ (1).xlsx


## 4. Full comparison table

Side-by-side Amila vs Experimental with per-mil differences.

In [14]:
comparison_df = compare_amila_to_experimental(amila_df, exp_rpfr_df)
comparison_df.head(10)

,Molecule,Method,Translational (Amila),Translational (Exp),Translational (Diff ‰),ZPE (total) (Amila),ZPE (total) (Exp),ZPE (total) (Diff ‰),Excited (harmonic) (Amila),Excited (harmonic) (Exp),Excited (harmonic) (Diff ‰),Excited (anharmonic) (Amila),Excited (anharmonic) (Exp),Excited (anharmonic) (Diff ‰),Rotational (diatomic) (Amila),Rotational (diatomic) (Exp),Rotational (diatomic) (Diff ‰),Rotational-vibrational (Amila),Rotational-vibrational (Exp),Rotational-vibrational (Diff ‰),RPFR 273.15K (Amila),RPFR 273.15K (Exp),RPFR 273.15K (Diff ‰)
0,1H2H/1H1H,CCSD(T)_Q5,1.835706,1.835706,-0.000184,4.527348,4.525226,0.468929,1.000000,1.000000,-0.000002,1.000000,1.000000,0.000002,2.594175,2.587616,2.534588,1.000000,1.000000,-0.000000,21.559880,21.495294,3.004668
1,1H3H/1H1H,CCSD(T)_Q5,2.820616,2.820616,0.000137,7.915773,7.890525,3.199725,1.000000,1.000000,-0.000006,1.000000,1.000000,0.000005,2.891130,2.881129,3.471349,1.000000,1.000000,-0.000000,64.551280,64.122800,6.682183
2,2H19F/1H19F,CCSD(T)_Q5,1.076388,1.076388,-0.000157,18.985190,19.003721,-0.975107,1.000000,1.000000,-0.000138,1.000000,1.000000,-0.000034,1.869562,1.860857,4.678185,1.000000,1.000000,-0.000002,38.205310,38.064531,3.698424
3,3H19F/1H19F,CCSD(T)_Q5,1.154288,1.154288,0.000405,67.935280,67.816399,1.752983,1.000002,1.000002,0.000174,0.999999,0.999999,-0.000074,2.656616,2.642312,5.413369,1.000000,1.000000,-0.000021,208.323600,206.839457,7.175339
4,2H35Cl/1H35Cl,CCSD(T)_Q5,1.042247,1.042247,-0.000384,9.003777,8.969002,3.877202,1.000012,1.000012,-0.000238,0.999996,0.999997,-0.000035,1.926513,1.918632,4.107506,1.000000,1.000000,-0.000128,18.078870,17.935379,8.000452
5,3H35Cl/1H35Cl,CCSD(T)_Q5,1.084888,1.084888,-0.000059,23.544280,23.413676,5.578093,1.000084,1.000086,-0.002161,0.999983,0.999983,-0.000236,2.800328,2.800915,-0.209685,1.000001,1.000001,0.000248,71.533400,71.151644,5.365385
6,1H37Cl/1H35Cl,CCSD(T)_Q5,1.084409,1.084409,-0.000344,1.005890,1.005875,0.015084,1.000000,1.000000,-0.000002,1.000000,1.000000,0.000001,1.001487,1.001488,-0.000808,1.000000,1.000000,-0.000000,1.092419,1.092403,0.014679
7,2H37Cl/1H35Cl,CCSD(T)_Q5,1.127798,1.127798,0.000181,9.078305,9.043813,3.813911,1.000012,1.000012,-0.000448,0.999996,0.999996,-0.000087,1.932141,1.932448,-0.158876,1.000000,1.000000,-0.000131,19.782380,19.710360,3.653911
8,3H37Cl/1H35Cl,CCSD(T)_Q5,1.171560,1.171560,0.000302,23.779840,23.647585,5.592740,1.000086,1.000088,-0.001912,0.999983,0.999983,-0.000266,2.812307,2.812904,-0.212136,1.000001,1.000001,0.000234,78.354920,77.935826,5.377430
9,35Cl37Cl/35Cl35Cl,CCSD(T)_Q5,1.043136,1.043136,-0.000352,1.020339,1.020151,0.184282,1.002218,1.002269,-0.050481,0.999973,0.999971,0.001163,2.055512,2.055500,0.005733,1.000006,1.000005,0.001071,2.192593,2.192284,0.140774


## 5. Per-molecule detailed comparison

Change `MOLECULE` below to inspect any isotopologue pair.

In [15]:
# --- Change this to inspect different pairs ---
MOLECULE = "1H2H/1H1H"  
# -----------------------------------------------

row_c = comparison_df[comparison_df["Molecule"] == MOLECULE]
if row_c.empty:
    print(f"Molecule '{MOLECULE}' not found. Available:")
    print(comparison_df["Molecule"].tolist())
else:
    # Reshape for readability
    records = []
    for col in AMILA_CONTRIB_COLS:
        records.append({
            "Contribution": col,
            "Amila": row_c[f"{col} (Amila)"].values[0],
            "Experimental": row_c[f"{col} (Exp)"].values[0],
            "Diff (\u2030)": row_c[f"{col} (Diff \u2030)"].values[0],
        })
    detail = pd.DataFrame(records).set_index("Contribution")
    print(f"=== {MOLECULE} ({row_c['Method'].values[0]}) ===")
    display(detail)

=== 1H2H/1H1H (CCSD(T)_Q5) ===


,Amila,Experimental,Diff (‰)
Contribution,,,
Translational,1.835706,1.835706,-0.000184
ZPE (total),4.527348,4.525226,0.468929
Excited (harmonic),1.000000,1.000000,-0.000002
Excited (anharmonic),1.000000,1.000000,0.000002
Rotational (diatomic),2.594175,2.587616,2.534588
Rotational-vibrational,1.000000,1.000000,-0.000000
RPFR 273.15K,21.559880,21.495294,3.004668


## 6. Summary: RPFR differences across all molecules

Grouped by parent molecule showing mean and max per-mil RPFR difference.

In [16]:
# Extract parent molecule from NSI label
def get_parent(mol_pair):
    """Extract parent molecule family from the pair label."""
    nsi = mol_pair.split("/")[1] if "/" in mol_pair else mol_pair
    # Map isotopologue labels to human-readable parent names
    from richet_rpfr.isotopes import parse_isotopologue_label, isotopologue_to_formula
    return isotopologue_to_formula(nsi)

summary = comparison_df[["Molecule", "Method", "RPFR 273.15K (Diff \u2030)"]].copy()
summary["Parent"] = summary["Molecule"].apply(get_parent)
summary = summary.rename(columns={"RPFR 273.15K (Diff \u2030)": "RPFR Diff (\u2030)"})

# Per-parent summary
parent_summary = (
    summary.groupby(["Method", "Parent"])["RPFR Diff (\u2030)"]
    .agg(["mean", "min", "max", "count"])
    .rename(columns={"mean": "Mean (\u2030)", "min": "Min (\u2030)", "max": "Max (\u2030)", "count": "N pairs"})
    .reset_index()
)
display(parent_summary)

,Method,Parent,Mean (‰),Min (‰),Max (‰),N pairs
0,CCSD(T)_DTQ,CaO,-0.191508,-0.357030,-0.025286,17
1,CCSD(T)_DTQ,CaS,0.204574,0.062891,0.424350,23
2,CCSD(T)_DTQ,KCl,0.016400,-0.012931,0.042828,5
3,CCSD(T)_DTQ,KF,-0.012001,-0.012001,-0.012001,1
4,CCSD(T)_DTQ,MgO,1.988802,0.578159,3.500520,8
5,CCSD(T)_DTQ,MgS,0.271877,0.064070,0.529665,11
6,CCSD(T)_DTQ,NaCl,-0.016891,-0.016891,-0.016891,1
7,CCSD(T)_Q5,CO,0.594613,0.196623,1.052147,8
8,CCSD(T)_Q5,CS,0.786281,0.093182,1.455342,11
9,CCSD(T)_Q5,Cl2,0.140774,0.140774,0.140774,1


## 7. Detailed contribution differences by parent molecule

Shows per-mil differences for every contribution, grouped by parent.

In [17]:
# --- Change this to filter by parent molecule ---
PARENT = "HF"  # e.g. "H2", "HF", "HCl", "CO", "N2", "Cl2", etc.
# -------------------------------------------------

diff_cols = [c for c in comparison_df.columns if c.endswith("(Diff \u2030)")]
short_names = [c.replace(" (Diff \u2030)", "") for c in diff_cols]

comparison_df["Parent"] = comparison_df["Molecule"].apply(get_parent)
subset = comparison_df[comparison_df["Parent"] == PARENT][["Molecule", "Method"] + diff_cols].copy()
subset.columns = ["Molecule", "Method"] + short_names

if subset.empty:
    parents_avail = sorted(comparison_df["Parent"].unique())
    print(f"Parent '{PARENT}' not found. Available: {parents_avail}")
else:
    print(f"=== Per-mil differences for {PARENT} isotopologues ===")
    display(subset.set_index("Molecule"))

=== Per-mil differences for HF isotopologues ===


,Method,Translational,ZPE (total),Excited (harmonic),Excited (anharmonic),Rotational (diatomic),Rotational-vibrational,RPFR 273.15K
Molecule,,,,,,,,
2H19F/1H19F,CCSD(T)_Q5,-0.000157,-0.975107,-0.000138,-0.000034,4.678185,-0.000002,3.698424
3H19F/1H19F,CCSD(T)_Q5,0.000405,1.752983,0.000174,-0.000074,5.413369,-0.000021,7.175339


## 8. All pairs - sorted by largest RPFR difference

In [18]:
ranked = summary.sort_values("RPFR Diff (\u2030)", key=abs, ascending=False)
display(ranked.head(20))

,Molecule,Method,RPFR Diff (‰),Parent
42,36S17O/32S16O,CCSD(T)_Q5,-91.786985,SO
4,2H35Cl/1H35Cl,CCSD(T)_Q5,8.000452,HCl
3,3H19F/1H19F,CCSD(T)_Q5,7.175339,HF
1,1H3H/1H1H,CCSD(T)_Q5,6.682183,H2
8,3H37Cl/1H35Cl,CCSD(T)_Q5,5.377430,HCl
5,3H35Cl/1H35Cl,CCSD(T)_Q5,5.365385,HCl
20,18O18O/16O16O,CCSD(T)_Q5,4.184505,O2
2,2H19F/1H19F,CCSD(T)_Q5,3.698424,HF
7,2H37Cl/1H35Cl,CCSD(T)_Q5,3.653911,HCl
81,26Mg18O/24Mg16O,CCSD(T)_DTQ,3.500520,MgO
